# 阶段 3：训练 0.5B / 1.5B 调度模型（Colab）

这个 notebook 每次跑一种配置。**只改第 3 格最上面两行**，然后「代码执行程序 → 全部运行」。

| 第几次 | `RUN` 填 | 做什么 | L4 上预计（估算） |
|---|---|---|---|
| 1 | `sweep_0.5b` | 0.5B 各训 8000 条，T = 0.1 / 0.3 / 1.0 和硬标签各一次，在验证集上比 | 40–60 分钟 |
| 2 | `main_0.5b` | 用选出的 T 正式训 0.5B（4 万条），评测，导出 GGUF | 40–60 分钟 |
| 3 | `soft_1.5b` | 同样的 T 训 1.5B，评测，导出 GGUF | 1–1.5 小时 |
| 4 | `hard_1.5b` | 1.5B 硬标签（只给最优那个答案），消融对照，也是阶段 4 GRPO 的起点 | 1–1.5 小时 |
| 可选 | `ps_0.5b` | 平分算力版成本做标签的 0.5B | 40–60 分钟 |

跑之前：菜单「代码执行程序 → 更改运行时类型」选 **L4 GPU**；Google Drive 里要有 `MyDrive/COMLLM/s3_data/`（Mac 上 `python -m train.make_data` 生成的 6 个文件）。

**结果**都存在 Drive 的 `MyDrive/COMLLM/s3_runs/<RUN>_<UTC 时间>/`，断线也不会丢。

**止损线**：
- 任何一格报错就停，把报错截图发到对话里，不要反复重跑（会一直扣额度）。
- 训练开始 20 步后会打印预计总时长。超过 2.5 小时就停下来告诉我，我们减条数。
- 跑完记得「代码执行程序 → 断开连接并删除运行时」，停止扣额度。

In [ ]:
# 第 1 格：安装固定版本的库（版本固定，以后复现时环境一样）
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.20.2", "peft==0.21.2", "accelerate", "sentencepiece"], check=True)
    # Colab 预装的 torchao 版本太老，peft 0.21 检测到它会直接报错。这里用不到 torchao，卸掉
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

import importlib, importlib.util
importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None, "torchao 还在：菜单「代码执行程序 → 重新启动会话」后再「全部运行」"
print("依赖安装完成")

In [ ]:
# 第 2 格：检查 GPU。L4 / A100 支持 bf16；T4 不支持，这个 notebook 不在 T4 上跑
import os, time, json, math, hashlib, csv, shutil
from datetime import datetime, timezone
import numpy as np
import torch

if IN_COLAB:
    subprocess.run(["nvidia-smi"], check=False)
    assert torch.cuda.is_available(), "没检测到 GPU：菜单「代码执行程序 → 更改运行时类型」选 L4 GPU 后再运行"
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
if HAS_GPU:
    assert torch.cuda.is_bf16_supported(), "这块 GPU 不支持 bf16（T4 就不支持）：换成 L4"
DTYPE = torch.bfloat16 if HAS_GPU else torch.float32
print("GPU:", torch.cuda.get_device_name() if HAS_GPU else "无（CPU）", "| 精度:", "bf16" if HAS_GPU else "fp32")
print("torch", torch.__version__, "（第一次跑时把这个版本号记进 docs/versions.md）")

In [ ]:
# 第 3 格：这次跑哪个配置。每次只改下面两行
RUN = "sweep_0.5b"     # 依次：sweep_0.5b → main_0.5b → soft_1.5b → hard_1.5b；可选 ps_0.5b
T_BEST = 0.3           # sweep_0.5b 跑完后，改成表里"多花延迟"最小的那个软标签 T

# ---- 下面不用改 ----
MODELS = {   # 名字: (Hugging Face 上的模型, 钉死的提交号，见 docs/versions.md)
    "0.5B": ("Qwen/Qwen2.5-0.5B-Instruct", "7ae557604adf67be50417f59c2c2f167def9a775"),
    "1.5B": ("Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"),
}
PRESETS = {
    # label：soft = 软标签 softmax(−成本/T)；hard = 只给最优动作
    # cost：paper = 论文单步公式；ps = 平分算力版（可选消融）
    "sweep_0.5b": {"model": "0.5B", "n_train": 8000, "export": False,
                   "sweep": [("soft", 0.1), ("soft", 0.3), ("soft", 1.0), ("hard", 0.0)]},
    "main_0.5b": {"model": "0.5B", "label": "soft", "T": T_BEST, "cost": "paper", "n_train": 40000, "export": True},
    "soft_1.5b": {"model": "1.5B", "label": "soft", "T": T_BEST, "cost": "paper", "n_train": 40000, "export": True},
    "hard_1.5b": {"model": "1.5B", "label": "hard", "T": 0.0, "cost": "paper", "n_train": 40000, "export": True},
    "ps_0.5b": {"model": "0.5B", "label": "soft", "T": T_BEST, "cost": "ps", "n_train": 40000, "export": True},
}
P = dict(PRESETS[RUN])

# 训练超参数：所有配置共用
LR = 2e-4                  # 学习率（LoRA 常用量级）
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
MICRO_BATCH = {"0.5B": 16, "1.5B": 8}   # 每次前向的条数
ACCUM = {"0.5B": 1, "1.5B": 2}          # 攒几次梯度再更新一次：两种模型每次更新都是 16 条
WARMUP_FRAC = 0.03         # 前 3% 的步数学习率从 0 线性升上去
EVAL_EVERY = 500           # 每多少次更新在验证集上看一眼
N_VAL_QUICK = 400          # 训练中途看验证集时只用前 400 条，省时间
SEED = 0
LLAMA_CPP_COMMIT = "5aeac37a3"          # 和 Jetson 上编译的 llama.cpp 是同一个版本

# 本机测试用（Colab 上这些环境变量都不存在，不起作用）
LOCAL_MODEL = os.environ.get("S3_LOCAL_MODEL")
if os.environ.get("S3_RUN"):
    RUN = os.environ["S3_RUN"]
    P = dict(PRESETS[RUN])
if os.environ.get("S3_N_TRAIN"):
    P["n_train"] = int(os.environ["S3_N_TRAIN"])
MAX_STEPS = int(os.environ.get("S3_MAX_STEPS", "0"))   # 0 = 不限制

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/COMLLM/s3_data"
    OUT = f"/content/drive/MyDrive/COMLLM/s3_runs/{RUN}_{STAMP}"
    WORK = "/content/work"
else:
    DATA_DIR = os.environ["S3_DATA"]
    OUT = os.path.join(os.environ["S3_OUT"], f"{RUN}_{STAMP}")
    WORK = os.environ.get("S3_WORK", os.path.join(os.environ["S3_OUT"], "work"))
os.makedirs(OUT, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print("这次跑:", RUN, P)
print("结果目录:", OUT)

In [ ]:
# 第 4 格：读数据，核对校验值，拼出提示词全文
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


with open(os.path.join(DATA_DIR, "meta.json"), encoding="utf-8") as f:
    META = json.load(f)
DATA = {}
for name, info in META["files"].items():
    path = os.path.join(DATA_DIR, name + ".jsonl")
    assert md5_of(path) == info["md5"], f"{name}.jsonl 的校验值和 meta.json 对不上：重新上传 data/s3 整个文件夹"
    DATA[name] = read_jsonl(path)
    for i, r in enumerate(DATA[name]):
        r["_idx"] = i                      # 在文件里是第几行，preds.csv 用它对应
    print(f"{name:13s} {len(DATA[name]):6d} 条 ✓")

SYSTEM = META["compact_system"]
LETTERS = META["letters"]                  # A = 本地，B = 服务器 1，……
EVAL_SPLITS = [s for s in ("val", "test_seen", "test_unseen", "friend_clean") if s in DATA]


def chatml(messages):
    """和仓库里 deploy/prompts.py 的 chatml 一模一样：Jetson 上就是这样拼提示词的。"""
    text = ""
    for m in messages:
        text = text + "<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>\n"
    text = text + "<|im_start|>assistant\n"
    return text


def compact_messages(row):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": row["user"]}]


for split in DATA:
    for r in DATA[split]:
        r["text"] = chatml(compact_messages(r))
print("提示词样例：")
print(DATA["train"][0]["text"])

In [ ]:
# 第 5 格：载入分词器，做三项核对（任何一项不过都会直接报错停下）
#   1. 12 个动作字母各是 1 个 token（单 token 打分的前提）
#   2. Hugging Face 的对话模板拼出来的文字 = 上面 chatml 拼的 = Mac 上用仓库代码拼的（meta.json 里的样例）
#   3. 答案字母紧跟在提示词后面，单独成 1 个 token，不会和前面的换行粘在一起
from transformers import AutoModelForCausalLM, AutoTokenizer

if LOCAL_MODEL:
    MODEL_ID, REVISION = LOCAL_MODEL, None
else:
    MODEL_ID, REVISION = MODELS[P["model"]]
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)

LETTER_IDS = []
for c in LETTERS:
    ids = tok.encode(c, add_special_tokens=False)
    assert len(ids) == 1, f"动作 {c} 不是单 token：{ids}"
    LETTER_IDS.append(ids[0])

for i, golden in enumerate(META["golden"]["compact"]):
    r = DATA["train"][i]
    assert r["text"] == golden, "这里拼的提示词和 Mac 上仓库代码拼的不一样"
    hf = tok.apply_chat_template(compact_messages(r), tokenize=False, add_generation_prompt=True)
    assert hf == golden, "Hugging Face 对话模板拼的提示词和 Mac 上的不一样"
rng_check = np.random.default_rng(0)
for j in rng_check.choice(len(DATA["train"]), size=min(200, len(DATA["train"])), replace=False):
    r = DATA["train"][int(j)]
    assert tok.apply_chat_template(compact_messages(r), tokenize=False, add_generation_prompt=True) == r["text"]

for r in DATA["train"][:50]:
    p_ids = tok(r["text"], add_special_tokens=False)["input_ids"]
    a = int(np.argmin(r["costs_compact"]))
    f_ids = tok(r["text"] + LETTERS[a], add_special_tokens=False)["input_ids"]
    assert f_ids == p_ids + [LETTER_IDS[a]], "答案字母和提示词末尾粘在了一起"


def encode_rows(rows):
    """把每行的提示词全文切成 token 编号，存进 r["ids"]。"""
    texts = [r["text"] for r in rows]
    ids = tok(texts, add_special_tokens=False)["input_ids"]
    for r, x in zip(rows, ids):
        r["ids"] = x


for split in EVAL_SPLITS:
    encode_rows(DATA[split])
lens = [len(r["ids"]) for r in DATA["test_seen"]]
print("三项核对都通过 ✓  动作字母的 token 编号:", dict(zip(LETTERS, LETTER_IDS)))
print(f"提示词长度（测试集）：平均 {np.mean(lens):.0f} token，最短 {min(lens)}，最长 {max(lens)}")

In [ ]:
# 第 6 格：训练和评测要用的函数（这一格只定义函数，不跑东西）
import torch.nn.functional as F


def teacher_probs(row, label, T, cost):
    """一条数据的目标分布，长度 12（超出动作个数的位置是 0）。

    1. 取成本：paper = 论文公式按压缩精度算的；ps = 平分算力版
    2. hard：成本最小的动作概率为 1，其余为 0
    3. soft：p_a = exp(−(c_a − c_min) / T) 再归一化。减掉 c_min 不改变结果，只是防止 exp 下溢
    """
    c = row["costs_compact"] if cost == "paper" else row["ps_costs"]
    n = len(c)
    p = [0.0] * len(LETTERS)
    best = 0
    for a in range(n):
        if c[a] < c[best]:
            best = a
    if label == "hard":
        p[best] = 1.0
        return p
    total = 0.0
    for a in range(n):
        p[a] = math.exp(-(c[a] - c[best]) / T)
        total += p[a]
    for a in range(n):
        p[a] = p[a] / total
    return p


def make_batches(rows, batch_size, rng, shuffle):
    """按服务器台数分组再切成批：同一批的提示词长度几乎一样，补齐浪费最少。

    1. 按 servers 把行分组
    2. 每组内（要打乱就先打乱）每 batch_size 条切成一批
    3. 要打乱就把所有批的顺序也打乱
    """
    groups = {}
    for r in rows:
        groups.setdefault(r["servers"], []).append(r)
    batches = []
    for k in sorted(groups):
        g = groups[k]
        if shuffle:
            order = rng.permutation(len(g))
            g = [g[int(i)] for i in order]
        for s in range(0, len(g), batch_size):
            batches.append(g[s:s + batch_size])
    if shuffle:
        order = rng.permutation(len(batches))
        batches = [batches[int(i)] for i in order]
    return batches


def collate(batch):
    """左补齐：短的提示词在左边补 pad，所有行的最后一个位置都是"assistant\n"之后、该出答案的地方。

    position_ids 让每行真实的第一个 token 从 0 开始编号，和 Jetson 上一条一条推理时完全一样。
    """
    width = max(len(r["ids"]) for r in batch)
    ids = torch.full((len(batch), width), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(batch), width), dtype=torch.long)
    for i, r in enumerate(batch):
        n = len(r["ids"])
        ids[i, width - n:] = torch.tensor(r["ids"], dtype=torch.long)
        mask[i, width - n:] = 1
    pos = (mask.cumsum(dim=1) - 1).clamp(min=0)
    return ids.to(DEVICE), mask.to(DEVICE), pos.to(DEVICE)


def last_logits(model, batch):
    """前向一次，只取最后一个位置的 logits（整个词表约 15 万个分数）。

    logits_to_keep=1 让模型只在最后一个位置算词表分数：其他位置用不到，省显存也省时间。
    """
    ids, mask, pos = collate(batch)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=HAS_GPU):
        out = model(input_ids=ids, attention_mask=mask, position_ids=pos, logits_to_keep=1)
    return out.logits[:, -1, :].float()


@torch.no_grad()
def predict(model, rows, batch_size):
    """按部署规则选动作，返回 {行号: (动作, 领先幅度, 合法字母总概率, 各合法字母概率)}。

    部署规则（和 deploy/prompts.py 一样）：在整个词表概率最高的 20 个 token 里，
    找概率最大的合法字母（A 到第 n_actions 个字母）；20 个里一个合法字母都没有就算非法，动作记 −1。
    领先幅度 = 选中字母和第二名合法字母的 log 概率之差，越小越容易被量化误差翻过来。
    """
    model.eval()
    out = {}
    for batch in make_batches(rows, batch_size, None, shuffle=False):
        logits = last_logits(model, batch)
        probs = torch.softmax(logits, dim=-1)
        p20 = torch.topk(probs, 20, dim=-1).values[:, -1]           # 每行第 20 名的概率
        letter_p = probs[:, LETTER_IDS].cpu().numpy()
        p20 = p20.cpu().numpy()
        for i, r in enumerate(batch):
            n = r["n_actions"]
            lp = letter_p[i, :n]
            best = int(np.argmax(lp))
            second = 0.0
            for a in range(n):
                if a != best and lp[a] > second:
                    second = float(lp[a])
            margin = math.log(max(float(lp[best]), 1e-30)) - math.log(max(second, 1e-30))
            pred = best if lp[best] >= p20[i] else -1
            out[r["_idx"]] = (pred, margin, float(lp.sum()), [round(float(x), 5) for x in lp])
    return out


def quick_metrics(rows, preds):
    """训练中途和跑完时看一眼用：准确率、平均多花延迟（槽）、非法率。正式指标在 Mac 上用 eval.open_loop 算。"""
    hit = 0
    regret = 0.0
    n_valid = 0
    invalid = 0
    for r in rows:
        a = preds[r["_idx"]][0]
        if a < 0:
            invalid += 1
            continue
        n_valid += 1
        regret += r["costs"][a] - r["costs"][r["best"]]
        if a == r["best"]:
            hit += 1
    return {"acc_pct": round(100.0 * hit / len(rows), 2),
            "regret": round(regret / n_valid, 4) if n_valid else None,
            "invalid_pct": round(100.0 * invalid / len(rows), 2)}


def write_preds(path, preds_by_split):
    """每条一行：split, idx, pred（−1 = 非法）, margin, letter_mass。Mac 上的 eval.open_loop 读这个文件。"""
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["split", "idx", "pred", "margin", "letter_mass"])
        for split, preds in preds_by_split.items():
            for idx in sorted(preds):
                pred, margin, mass, _ = preds[idx]
                w.writerow([split, idx, pred, round(margin, 4), round(mass, 4)])


def lr_at(step, total):
    """学习率：前 WARMUP_FRAC 的步数从 0 线性升到 LR，之后按余弦曲线降到 0。"""
    warm = max(1, int(total * WARMUP_FRAC))
    if step < warm:
        return LR * (step + 1) / warm
    progress = (step - warm) / max(1, total - warm)
    return LR * 0.5 * (1.0 + math.cos(math.pi * progress))


def load_base():
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=DTYPE)
    return model.to(DEVICE)


def run_label(label, T, cost):
    name = P["model"].lower() + "_" + label
    if label == "soft":
        name += f"_T{T:g}"
    if cost == "ps":
        name += "_ps"
    return name

In [ ]:
# 第 7 格：训练函数（这一格只定义函数，第 8 格才真正开始训练）
from peft import LoraConfig, get_peft_model


def train_one(model, label, T, cost, n_train, log_path):
    """在 model 上套 LoRA 训练，返回训练好的模型（还带着 LoRA，没有合并）。

    1. 套上 LoRA：只训练每个线性层旁边的两个小矩阵，原来的权重不动
    2. 从训练集里随机取 n_train 条，算好每条的目标分布
    3. 按服务器台数分批，打乱
    4. 每批：前向（只要最后一个位置）→ 取 12 个字母的 log 概率 → KL 损失 → 反向
       攒够 ACCUM 批更新一次：梯度裁剪到 1.0 → 设学习率 → AdamW 更新
    5. 每 EVAL_EVERY 次更新在验证集前 N_VAL_QUICK 条上看准确率和多花延迟，写进训练日志
    """
    # 1. LoRA
    cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                     target_modules="all-linear", task_type="CAUSAL_LM")
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()

    # 2. 训练数据
    rng = np.random.default_rng(SEED)
    order = rng.permutation(len(DATA["train"]))[:n_train]
    rows = [DATA["train"][int(i)] for i in order]
    encode_rows(rows)
    target = {}
    for r in rows:
        target[r["_idx"]] = teacher_probs(r, label, T, cost)

    # 3. 分批
    mb = MICRO_BATCH[P["model"]]
    acc = ACCUM[P["model"]]
    batches = make_batches(rows, mb, rng, shuffle=True)
    total_updates = len(batches) // acc
    if MAX_STEPS:
        total_updates = min(total_updates, MAX_STEPS)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    letter_ids = torch.tensor(LETTER_IDS, device=DEVICE)
    val_rows = DATA["val"][:N_VAL_QUICK]

    # 4. 训练循环
    log = open(log_path, "w", newline="", encoding="utf-8")
    logw = csv.writer(log)
    logw.writerow(["update", "loss", "lr", "elapsed_s", "val_acc_pct", "val_regret", "val_invalid_pct"])
    model.train()
    t0 = time.time()
    loss_sum = 0.0
    loss_n = 0
    update = 0
    for bi, batch in enumerate(batches):
        if update >= total_updates:
            break
        logits = last_logits(model, batch)                         # (条数, 词表)
        logp = torch.log_softmax(logits, dim=-1)[:, letter_ids]    # (条数, 12)：12 个字母的 log 概率
        p_t = torch.tensor([target[r["_idx"]] for r in batch], device=DEVICE)
        # KL(目标 ‖ 模型) = Σ_a p_t(a)·(log p_t(a) − log p_s(a))；p_t(a) = 0 的项按 0 算
        kl = torch.special.xlogy(p_t, p_t) - p_t * logp
        loss = kl.sum(dim=1).mean()
        (loss / acc).backward()
        loss_sum += loss.item()
        loss_n += 1
        if (bi + 1) % acc != 0:
            continue
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        lr = lr_at(update, total_updates)
        for g in opt.param_groups:
            g["lr"] = lr
        opt.step()
        opt.zero_grad(set_to_none=True)
        update += 1

        elapsed = time.time() - t0
        if update == 20:
            eta = elapsed / 20 * total_updates
            print(f"前 20 次更新用了 {elapsed:.0f} 秒，预计训练总共约 {eta / 60:.0f} 分钟"
                  + ("  ← 超过 2.5 小时，停下来告诉我" if eta > 2.5 * 3600 else ""))
        val = {"acc_pct": "", "regret": "", "invalid_pct": ""}
        if update % EVAL_EVERY == 0 or update == total_updates:
            val = quick_metrics(val_rows, predict(model, val_rows, mb * 2))
            model.train()
        if update % 50 == 0 or update == total_updates or val["acc_pct"] != "":
            avg = loss_sum / max(loss_n, 1)
            logw.writerow([update, round(avg, 5), f"{lr:.2e}", round(elapsed, 1),
                           val["acc_pct"], val["regret"], val["invalid_pct"]])
            log.flush()
            extra = ""
            if val["acc_pct"] != "":
                extra = f"  | 验证集 准确率 {val['acc_pct']}%  多花 {val['regret']} 槽  非法 {val['invalid_pct']}%"
            print(f"更新 {update}/{total_updates}  loss {avg:.4f}  学习率 {lr:.2e}  {elapsed / 60:.1f} 分钟{extra}")
            loss_sum = 0.0
            loss_n = 0
    log.close()
    print(f"训练完成，用时 {(time.time() - t0) / 60:.1f} 分钟")
    return model

In [ ]:
# 第 8 格：开始跑。sweep_0.5b 只训练和看验证集；其余配置还要评测全部测试集、存 LoRA
torch.manual_seed(SEED)
mb_eval = MICRO_BATCH[P["model"]] * 2
config = {"run": RUN, "preset": P, "model": MODEL_ID, "revision": REVISION, "model_tag": P["model"],
          "lr": LR, "lora": [LORA_R, LORA_ALPHA, LORA_DROPOUT], "micro_batch": MICRO_BATCH[P["model"]],
          "accum": ACCUM[P["model"]], "seed": SEED, "data_version": META["version"], "data_git": META["git"],
          "torch": torch.__version__, "started_utc": STAMP}

if "sweep" in P:
    results = []
    for label, T in P["sweep"]:
        name = run_label(label, T, "paper")
        print(f"\n===== {name} =====")
        model = train_one(load_base(), label, T, "paper", P["n_train"], f"{OUT}/train_log_{name}.csv")
        preds = predict(model, DATA["val"], mb_eval)
        m = quick_metrics(DATA["val"], preds)
        write_preds(f"{OUT}/preds_val_{name}.csv", {"val": preds})
        results.append({"name": name, "label": label, "T": T, **m})
        print(name, "验证集：", m)
        del model
        if HAS_GPU:
            torch.cuda.empty_cache()
    with open(f"{OUT}/sweep.json", "w", encoding="utf-8") as f:
        json.dump({"config": config, "results": results}, f, ensure_ascii=False, indent=2)
    print("\n| 配置 | 验证集准确率 % | 多花延迟（槽） | 非法率 % |")
    print("|---|---|---|---|")
    for x in results:
        print(f"| {x['name']} | {x['acc_pct']} | {x['regret']} | {x['invalid_pct']} |")
    soft = [x for x in results if x["label"] == "soft"]
    best = min(soft, key=lambda x: (x["regret"] if x["regret"] is not None else 1e9, -x["acc_pct"]))
    print(f"\n软标签里多花延迟最小的是 T = {best['T']:g}：把第 3 格的 T_BEST 改成 {best['T']:g}，RUN 改成 main_0.5b")
else:
    name = run_label(P["label"], P["T"], P["cost"])
    config["name"] = name
    # 1. 训练前先评测一次基座（对照"未训练"）
    model = load_base()
    preds_u = {}
    for split in EVAL_SPLITS:
        preds_u[split] = predict(model, DATA[split], mb_eval)
        print("未训练", split, quick_metrics(DATA[split], preds_u[split]))
    write_preds(f"{OUT}/preds_untrained.csv", preds_u)
    # 2. 训练
    model = train_one(model, P["label"], P["T"], P["cost"], P["n_train"], f"{OUT}/train_log.csv")
    # 3. 评测全部集合，存预测和 LoRA
    preds_all = {}
    quick = {}
    for split in EVAL_SPLITS:
        preds_all[split] = predict(model, DATA[split], mb_eval)
        quick[split] = quick_metrics(DATA[split], preds_all[split])
        print("训练后", split, quick[split])
    write_preds(f"{OUT}/preds.csv", preds_all)
    model.save_pretrained(f"{OUT}/adapter")
    config["quick_metrics"] = quick
    with open(f"{OUT}/config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    print("LoRA 和预测都存好了：", OUT)

In [ ]:
# 第 9 格：合并 LoRA → 转 GGUF（Q8_0 和 Q4_K_M），用和 Jetson 同一版本的 llama.cpp
def run_quiet(cmd, what):
    """跑一个外部命令，输出先收起来；成功只打一行，失败打最后 40 行再报错停下。"""
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        lines = (r.stdout + r.stderr).splitlines()
        print("\n".join(lines[-40:]))
        raise RuntimeError(f"{what} 失败（返回码 {r.returncode}），上面是最后 40 行输出")
    print(f"{what} 完成，用时 {time.time() - t0:.0f} 秒")


if P.get("export"):
    # 1. 合并：把 LoRA 的两个小矩阵乘起来加回原权重，得到一个普通的模型
    merged = model.merge_and_unload()
    merged.eval()
    MERGED_DIR = f"{WORK}/merged_{name}"
    merged.save_pretrained(MERGED_DIR)
    tok.save_pretrained(MERGED_DIR)
    # 2. 取 llama.cpp 源码，切到 Jetson 上同一个提交
    LCPP = os.environ.get("S3_LLAMA_CPP", f"{WORK}/llama.cpp")
    if not os.path.isdir(LCPP):
        run_quiet(["git", "clone", "-q", "https://github.com/ggml-org/llama.cpp", LCPP], "下载 llama.cpp 源码")
    run_quiet(["git", "-C", LCPP, "checkout", "-q", LLAMA_CPP_COMMIT], "切到提交 " + LLAMA_CPP_COMMIT)
    # 3. Python 转换脚本直接出 Q8_0；Q4_K_M 要先出 F16，再用 llama-quantize 量化
    GGUF_DIR = f"{OUT}/gguf"
    os.makedirs(GGUF_DIR, exist_ok=True)
    tag = "comllm-" + name.replace("_", "-")
    q8 = f"{GGUF_DIR}/{tag}-q8_0.gguf"
    f16 = f"{WORK}/{tag}-f16.gguf"
    q4 = f"{GGUF_DIR}/{tag}-q4_k_m.gguf"
    run_quiet([sys.executable, f"{LCPP}/convert_hf_to_gguf.py", MERGED_DIR, "--outtype", "q8_0", "--outfile", q8], "转 Q8_0")
    run_quiet([sys.executable, f"{LCPP}/convert_hf_to_gguf.py", MERGED_DIR, "--outtype", "f16", "--outfile", f16], "转 F16")
    quant = f"{LCPP}/build/bin/llama-quantize"
    if not os.path.exists(quant):
        # 只编译 llama-quantize 这一个程序，用 CPU，不需要 CUDA，约 3–5 分钟（估算）
        run_quiet(["cmake", "-S", LCPP, "-B", f"{LCPP}/build", "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF",
                   "-DCMAKE_BUILD_TYPE=Release"], "配置编译")
        run_quiet(["cmake", "--build", f"{LCPP}/build", "--target", "llama-quantize", "-j", str(os.cpu_count() or 2)],
                  "编译 llama-quantize")
    run_quiet([quant, f16, q4, "Q4_K_M"], "量化成 Q4_K_M")
    os.remove(f16)
    for p in (q8, q4):
        print(f"{os.path.basename(p)}  {os.path.getsize(p) / 1e6:.0f} MB")
else:
    print("这个配置不导出 GGUF，跳过")

In [ ]:
# 第 10 格：出 Jetson 比对用的 200 条（合并后的模型在 PyTorch 上选的动作）
if P.get("export"):
    # 1. 从两个测试集里按服务器台数轮流取，共 200 条，9 种台数都有
    groups = {}
    for split in ("test_seen", "test_unseen"):
        for r in DATA[split]:
            groups.setdefault(r["servers"], []).append(r)
    picked = []
    pos = {k: 0 for k in groups}
    while len(picked) < 200:
        for k in sorted(groups):
            g = groups[k]
            step = max(1, len(g) // 23)
            j = pos[k] * step
            if j < len(g) and len(picked) < 200:
                picked.append(g[j])
            pos[k] += 1
    # 2. 用合并后的模型（转 GGUF 的就是它）按部署规则选动作。
    #    两份测试集分开算：predict 按行号存结果，两份的行号会重复，放在一起算会互相覆盖
    pt = {}
    for split in ("test_seen", "test_unseen"):
        sub = []
        for r in picked:
            if r["split"] == split:
                sub.append(r)
        res = predict(merged, sub, mb_eval)
        for idx in res:
            pt[(split, idx)] = res[idx]
    JDIR = f"{OUT}/jetson_check"
    os.makedirs(JDIR, exist_ok=True)
    n_same_as_lora = 0
    with open(f"{JDIR}/expected_200.jsonl", "w", encoding="utf-8") as f:
        for r in picked:
            pred, margin, mass, lp = pt[(r["split"], r["_idx"])]
            if pred == preds_all[r["split"]][r["_idx"]][0]:
                n_same_as_lora += 1
            f.write(json.dumps({"split": r["split"], "idx": r["_idx"], "servers": r["servers"], "user": r["user"],
                                "state": r["state"], "n_actions": r["n_actions"], "costs": r["costs"],
                                "pt_action": pred, "pt_margin": margin, "pt_letter_probs": lp},
                               ensure_ascii=False) + "\n")
    margins = []
    for r in picked:
        margins.append(pt[(r["split"], r["_idx"])][1])
    margins.sort()
    print(f"写好了 {JDIR}/expected_200.jsonl")
    print(f"合并前后选的动作一致 {n_same_as_lora}/200（合并只是把同样的数加回去，应该是 200 或差一两条）")
    print(f"领先幅度：最小 {margins[0]:.3f}，中位数 {margins[100]:.3f}；小于 0.1 的有 {sum(m < 0.1 for m in margins)} 条，"
          f"这些最容易被量化翻过来")
print("\n全部完成 ✓ 结果目录：", OUT)
print(sorted(os.listdir(OUT)))

## 跑完之后

1. 把第 8 格最后的表（sweep）或"训练后"那几行（其他配置）截图发到对话里。
2. 按运行文档把结果目录下载到 Mac 的 `runs/s3/` 里。GGUF 文件不进仓库，拷到 Jetson 上比对。
3. 菜单「代码执行程序 → 断开连接并删除运行时」，停止扣额度。